# Yoga Instruction Evaluation Metrics

This notebook evaluates the fine-tuned Qwen model on the yoga instruction test set using BERTScore and qualitative analysis.

In [1]:
# Install necessary libraries if not already installed
!pip install bert_score pandas peft

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import json
import pandas as pd
from bert_score import score
from tqdm import tqdm
import os

/home/phsuanh/miniconda3/envs/huggingface/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Model and Tokenizer

In [3]:
# Paths
BASE_MODEL_PATH = "/home/phsuanh/Documents/Neuro.care/ai-yoga-assistant/LLM/Qwen3-0.6B_local"
ADAPTER_PATH = "/home/phsuanh/Documents/Neuro.care/ai-yoga-assistant/LLM/Qwen3-0.6B_local/qwen-yoga-finetune/checkpoint-264"
TEST_SET_PATH = "/home/phsuanh/Documents/Neuro.care/ai-yoga-assistant/backend/synthetic_yoga_instructions_10000_testset.json"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load Base Model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    attn_implementation="flash_attention_2",
    device_map="auto",
    trust_remote_code=True
)

# Load Adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
# Move to GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print("Model loaded successfully.")

Model loaded successfully.


## 2. Load Test Data

In [4]:
with open(TEST_SET_PATH, 'r') as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} test examples.")

# Use a subset for faster evaluation if needed, or full set
# dataset_subset = test_data[:100] # Uncomment to test on 100 examples
dataset_subset = test_data # Use full set

Loaded 10000 test examples.


## 3. Generate Predictions

In [ ]:
def generate_response(instruction, input_data, max_new_tokens=500):
    # 2. PROMPT FORMATTING
    # Combine instruction and input
    text_prompt = f"{instruction}\n\nInput: {input_data}\n\nResponse:"

    # 3. INPUT CONTROL (Truncation)
    # We define how much space we need to reserve for the generated answer.
    # If the model has a 2048 limit, and we want 128 tokens for output,
    # the input cannot be larger than 1920 tokens.
    max_input_length = 2048 - max_new_tokens

    inputs = tokenizer(
        text_prompt, 
        return_tensors="pt", 
        truncation=True,           # <--- Automatically cuts off excess input
        max_length=max_input_length 
    ).to(device)

    # 4. OUTPUT CONTROL (Generation)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # <--- Controls Output Length
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    # 5. DECODING
    # We slice the output to remove the input prompt and keep only the new text
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

predictions = []
references = []
inputs_list = []

# Run inference
print("Generating predictions...")
for example in tqdm(dataset_subset[:7]):
    instruction = "Analyze this yoga pose."
    input_text = example['input']
    target_text = example['output']
    
    prediction = generate_response(instruction, input_text)
    
    inputs_list.append(input_text)
    predictions.append(prediction)
    references.append(target_text)



Generating predictions...


  3%|▎         | 290/10000 [20:56<11:40:55,  4.33s/it]


KeyboardInterrupt: 

In [8]:
# Save results to DataFrame
df_results = pd.DataFrame({
    'Input': inputs_list,
    'Reference': references,
    'Prediction': predictions
})

df_results.head()

,Input,Reference,Prediction
0,Pose: Garland\nJoint: right_hip\nCurrent Angle...,"The position is correct, but your Garland lack...",I'm not seeing enough flex in your Right Hip....
1,Pose: Seated_Forward_Fold\nJoint: left_shoulde...,Focus on flexing your Left Shoulder to improve...,I'm not seeing enough straighten in your Left...
2,Pose: Chair\nJoint: left_shoulder\nCurrent Ang...,Fantastic progress! Let's extend that Left Sho...,I'm not sure how this Chair will work for you...
3,Pose: Seated_Forward_Fold\nJoint: left_knee\nC...,Beautiful Seated Forward Fold! Just straighten...,I'm not sure how you're bending this Seated F...
4,Pose: Half_Moon\nJoint: left_knee\nCurrent Ang...,Excellent effort on this Half Moon! I can see ...,Let's work on bending your Left Knee to impro...


## 4. Calculate BERTScore

In [9]:
print("Calculating BERTScore...")
P, R, F1 = score(predictions, references, lang="en", verbose=True)

print(f"BERTScore Precision: {P.mean():.4f}")
print(f"BERTScore Recall: {R.mean():.4f}")
print(f"BERTScore F1: {F1.mean():.4f}")

# Add scores to DataFrame
df_results['BERTScore_F1'] = F1.tolist()
df_results['BERTScore_P'] = P.tolist()
df_results['BERTScore_R'] = R.tolist()

Calculating BERTScore...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 9/9 [00:01<00:00,  5.15it/s]


computing greedy matching.


100%|██████████| 5/5 [00:00<00:00, 67.32it/s]

done in 1.83 seconds, 158.36 sentences/sec
BERTScore Precision: 0.8846
BERTScore Recall: 0.8990
BERTScore F1: 0.8914


## 5. Qualitative Evaluation (Human Eye-ball)

In [10]:
# Display random examples
pd.set_option('display.max_colwidth', None)
print("Random Examples for Human Evaluation:")
display(df_results.sample(5)[['Input', 'Reference', 'Prediction', 'BERTScore_F1']])

Random Examples for Human Evaluation:


,Input,Reference,Prediction,BERTScore_F1
224,Pose: Half_Moon\nJoint: right_hip\nCurrent Angle: 88.0°\nTarget Range: 52.7° - 82.7°\nDeviation: +20.3°\nConfidence: 80.7%,Excellent effort! Focus on extending your Right Hip for perfect alignment.,I'm not seeing enough stability in your Right Hip. Adjust it by flexing it in this Half Moon.,0.892254
255,Pose: Wheel\nJoint: right_knee\nCurrent Angle: 57.5°\nTarget Range: 27.9° - 57.9°\nDeviation: +14.5°\nConfidence: 76.3%,Great work on your Wheel! Try to straighten your Right Knee just a bit more.,I'm not sure how you're bending it. Let's make sure you're properly bending the Right Knee in this Wheel.,0.888223
215,Pose: Seated_Forward_Fold\nJoint: right_knee\nCurrent Angle: 9.2°\nTarget Range: 24.3° - 54.3°\nDeviation: -30.2°\nConfidence: 49.5%,Fantastic progress! Let's bend that Right Knee to refine your Seated Forward Fold.,Let's get this Seated Forward Fold strong. Extend your Right Knee by adjusting your alignment. This will help as it is a core exercise.,0.921385
269,Pose: Wheel\nJoint: right_knee\nCurrent Angle: 78.4°\nTarget Range: 97.4° - 127.4°\nDeviation: -34.0°\nConfidence: 35.9%,Beautiful Wheel! Just bend your Right Knee slightly to enhance the stretch.,Let's fix the position of your Right Knee in this Wheel. Focus on bending it correctly. This adjustment will help improve our overall form.,0.904884
50,Pose: Staff\nJoint: right_elbow\nCurrent Angle: 98.4°\nTarget Range: 96.9° - 126.9°\nDeviation: -13.5°\nConfidence: 87.5%,Adjust your Right Elbow by bending it slightly in your Staff.,I'm not seeing enough extension in your Right Elbow. Try straightening it to match the target form. This adjustment will help as it it builds proper alignment and stability.,0.859695


In [11]:
# Save results to CSV
df_results.to_csv("yoga_evaluation_results.csv", index=False)
print("Results saved to yoga_evaluation_results.csv")

Results saved to yoga_evaluation_results.csv
